# T08 - Milestone Project 2 Test

Covers course section **08-Milestone Project - 2** using the course notebooks
`08-Milestone Project - 2/00-Milestone-2-Warmup-Project.ipynb` (the simple War card
game with the `Card`, `Deck` and `Player` classes),
`08-Milestone Project - 2/01-Milestone Project 2 - Assignment.ipynb` (the Blackjack
milestone assignment) and
`08-Milestone Project - 2/02-Milestone Project 2 - Walkthrough Steps Workbook.ipynb`
(the step by step workbook with the `Card`, `Deck`, `Hand` and `Chips` classes), plus
the OOP material in `05-Object Oriented Programming/01-Object Oriented Programming.ipynb`
and the testing material in `07-Errors and Exception Handling/04-Unit Testing.ipynb`.

**10 questions / 20 points** - 8 design / rubric questions (2 pts each) and
2 coding exercises (2 pts each).

Suggested time budget: 60 minutes.

How the checks work: each of the two coding exercises has a code cell tagged `check`
that runs `assert` statements against the exact class, function and attribute names
your question asked you to create. Run the notebook from top to bottom; a check
passes when it raises no error. The design questions are answered in the collapsed
`Solution` blocks.

See `S0 - Testing Syllabus.ipynb` for the full course testing plan.

## Q1. Design question (2 points)

The warm-up project plays a simplified game of War with two players. List the rules of
the game that you would encode in your program, as described in the course notes.

<details>
<summary>Solution</summary>

**Answer:** The rules you would encode are:

1. Build a standard 52 card deck, shuffle it, and deal it out so that each of the
   two players starts with half of the deck (26 cards each).
2. In every round each player removes one card from their hand and the two cards are
   compared by value.
3. The player with the higher card value wins both cards and adds them to their own
   hand.
4. If the two cards have the same value the game is "at war": the round continues,
   both players put further cards down and the comparison is repeated.
5. If a player does not have enough cards left to play the war (the warm-up needs 5
   more cards each) that player cannot continue and loses the game.
6. The game ends as soon as one player has no cards left; the other player wins.

Card values follow the course mapping: Two to Ten are 2 to 10, Jack 11, Queen 12,
King 13 and Ace 14, so the Ace is the highest card.

</details>

## Q2. Design question (2 points)

How would you model the `Card`, `Deck` and `Player` classes for the War game with
classes? Name the attributes and the methods of each class.

<details>
<summary>Solution</summary>

**Answer:** **`Card`** - one card. Attributes: `suit` (Hearts, Diamonds,
Spades or Clubs), `rank` (Two ... Ace) and `value` (the number the card is worth,
taken from a `values` dictionary so Two is 2 and Ace is 14). Method: `__str__`, which
returns the rank and the suit, for example `"Two of Hearts"`.

**`Deck`** - the whole set of cards. Attribute: `all_cards`, a list that is filled in
`__init__` by looping over the suits and the ranks and creating a `Card` for every
combination, so the deck holds 52 `Card` objects. Methods: `shuffle()`, which shuffles
`all_cards` in place with `random.shuffle`, and `deal_one()`, which removes one card
from the deck (`self.all_cards.pop()`) and returns it.

**`Player`** - one participant. Attributes: `name` and `all_cards`, a list that starts
empty. Methods: `add_cards(new_cards)`, which accepts either a single card or a list
of cards and adds them to the hand, `remove_one()`, which removes and returns the
card at the top of the hand (`self.all_cards.pop(0)`), and `__str__`, which returns
something like `Player One has 26 cards.`.

The `Deck` class uses the `Card` class inside its own `__init__`, which is only
possible because the `Card` class is defined first.

</details>

## Q3. Design question (2 points)

You want to add a timer to the milestone game so that a player can see how long a
round or a decision takes. Would you write the timer as a class or as a function,
and would you use a background thread or a simple loop? Justify your answer.

<details>
<summary>Solution</summary>

**Answer:** Either choice is defensible, but for this project a simple
function or a small class driven by a plain loop is the better answer.

* **Function versus class.** A function such as `countdown(seconds)` is the smallest
  thing that works, but a small `Timer` class fits the milestone better because the
  project is built on OOP: the timer can then hold the remaining time as an attribute
  and be passed around like the other objects of the game. The rule of thumb from the
  course is that a class is worth it when the object carries state and has several
  methods.
* **Thread versus loop.** A `threading.Thread` is unnecessary here. A background
  thread needs to be started, stopped and joined, it makes the output order
  unpredictable and it is much harder to test. A `while` loop that calls
  `time.sleep(1)` once per second is simple, readable and deterministic enough for a
  card game.

Whatever you choose, keep the clock injectable (a `sleep` function or an explicit
`now` argument) so that the timer can be unit tested without waiting for real time to
pass.

</details>

## Q4. Design question (2 points)

How would you unit test the deck logic of your milestone project with `unittest`?
Name the test class, the test methods and what you would assert.

<details>
<summary>Solution</summary>

**Answer:** Follow the pattern of `07-Errors and Exception Handling/04-Unit Testing.ipynb`:
write the code under test in its own module (for example `deck.py`) and the tests in
a separate file (for example `test_deck.py`), then import both and let
`unittest.main()` run them.

A test class such as `class TestDeck(unittest.TestCase):` would contain:

* `test_deck_size` - a new deck holds 52 cards: `self.assertEqual(len(deck.all_cards), 52)`
* `test_cards_are_unique` - no card appears twice: the set of `(suit, rank)` pairs has
  length 52
* `test_card_value` - the value mapping is right, for example
  `self.assertEqual(Card('Hearts', 'Ace').value, 14)`
* `test_deal_one` - dealing removes exactly one card and returns a `Card`
* `test_shuffle_keeps_every_card` - shuffling does not lose or duplicate cards, and
  the deck still has 52 cards afterwards
* `test_deal_one_on_empty_deck` - dealing from an empty deck raises `IndexError`

Write the tests from simple to complex, because every test function is run in order.

</details>

## Q5. Design question (2 points)

How would you test a timer so that the test does not depend on real time passing?
Give a concrete approach with the `unittest` style used in the course.

<details>
<summary>Solution</summary>

**Answer:** Never assert on real elapsed wall clock time in a test. Instead
make the timer controllable and test the logic, not the clock.

A concrete approach:

* Give the timer an injectable dependency, for example
  `def countdown(seconds, sleep_func=time.sleep):` or
  `def elapsed_seconds(start, end=None):`. The default is the real `time.sleep` or
  `time.time`, but a test passes its own.
* In the test, pass a fake sleeper that only records the calls:
  `calls = []` and then `countdown(3, sleep_func=calls.append)`.
* Assert on the recorded calls and on the returned value, for example
  `self.assertEqual(calls, [1, 1, 1])` and
  `self.assertEqual(result, [3, 2, 1])`.
* For a function that measures elapsed time, pass both the start and the end value
  explicitly (`elapsed_seconds(100.0, 130.5)`) so the expected result is a constant.
* Also test the formatting separately, for example `self.assertEqual(format_time(185), '03:05')`.

This keeps the test fast, deterministic and readable, which is the whole point of
writing unit tests.

</details>

## Q6. Design question (2 points)

Which output formats would your game print? Give examples of the exact strings you
would use, based on the course material.

<details>
<summary>Solution</summary>

**Answer:** Based on the warm-up project and the course code, sensible formats
are:

* A card: `"Two of Hearts"`, produced by `Card.__str__` from the rank and the suit.
* A player: `Player One has 26 cards.`, produced by `Player.__str__` with
  `f'Player {self.name} has {len(self.all_cards)} cards.'`.
* The round counter: `Round 1`, `Round 2`, and so on.
* The war announcement: `WAR!`
* The end of the game: `Player One out of cards! Game Over` and `Player Two Wins!`
* The case where a player cannot play the war:
  `Player One unable to play war! Game Over at War` and
  `Player Two Wins! Player One Loses!`
* For the Blackjack version of the milestone, the hand totals and the chip total, for
  example `Total chips: 100`.

Keep the strings in one place (a constant or a small helper) so that the format is
consistent and easy to change.

</details>

## Q7. Design question (2 points)

Which errors can happen in this project and how should you handle them? Use the
`try` / `except` / `else` / `finally` notation of the course.

<details>
<summary>Solution</summary>

**Answer:** The realistic errors are:

* **Bad user input.** `int(input(...))` raises a `ValueError` when the player types
  something that is not a number. Handle it with `try` / `except ValueError` and
  ask again, the same way as the `askint()` function of the errors section.
* **A bet that is too large.** The bet must not exceed the available chips. Check it
  yourself and `raise ValueError("Your bet exceeds your chips")`, then catch it in the
  calling function so the game can continue.
* **An empty deck.** `deal_one()` on an empty deck raises an `IndexError` from
  `list.pop()`. Either check `len(deck.all_cards)` before dealing or catch the
  `IndexError` and end the game cleanly.
* **A wrong card value.** A `KeyError` from the `values` dictionary means the rank is
  not one you know about; use `values.get(rank, 0)` or validate the rank when the
  card is created.

A useful shape for the input handling is the course pattern:

```
while True:
    try:
        value = int(input("Please enter an integer: "))
    except ValueError:
        print("Looks like you did not enter an integer!")
        continue
    else:
        print("Yep that's an integer!")
        break
    finally:
        print("Finally, I executed!")
```

The `finally` block is the right place for cleanup such as closing a file.

</details>

## Q8. Design question (2 points)

Give three ideas for extending the milestone project beyond the requirements, and
say which classes or functions you would touch.

<details>
<summary>Solution</summary>

**Answer:** Possible extensions, each with the part of the project it
touches:

1. **More players.** The `Player` class and the setup code that deals the deck
   already work for any number of players, so generalise the loop that deals the
   cards instead of hard coding two players.
2. **Betting and chips in the War version.** Add a `Chips` class like the one in the
   walkthrough workbook, with `total`, `bet`, `win_bet()` and `lose_bet()`, and call
   it from the game loop after every round.
3. **Statistics.** Count how many rounds each player won and how many wars happened
   and print a summary at the end of the game; the counters can live in the `Player`
   class or in the game function.
4. **A cleaner deck object.** Add a `reset()` method to `Deck` so the game can be
   replayed without rebuilding the deck, and a `__str__` to `Deck` that prints the
   remaining cards, which the workbook suggests for troubleshooting.
5. **The Blackjack hand logic.** Add a `Hand` class with `add_card()` and
   `adjust_for_ace()` from the walkthrough workbook so that aces count as 11 or 1 and
   the hand value is calculated in one place.
6. **A save and load feature.** Write the final scores to a file with the `json`
   module and read them back the next time the program starts.

</details>

## Q9. Coding exercise (2 points)

Create the card and deck classes for the game:

* Import `random` at the top of the cell.
* Create a class **`Card`** whose `__init__(self, suit, rank)` stores `self.suit`,
  `self.rank` and `self.value`. The value is the number of the card, where `Two` is
  2, `Three` is 3, ... `Jack` is 11, `Queen` is 12, `King` is 13 and `Ace` is 14.
  Add a `__str__(self)` that returns the rank, the word `of` and the suit, for
  example `"Two of Hearts"`.
* Create a class **`Deck`** whose `__init__(self)` fills the list `self.all_cards`
  with the 52 cards of a standard deck, one card for each of the four suits and each
  of the thirteen ranks. Add a method `shuffle(self)` that shuffles `self.all_cards`
  in place with `random.shuffle`, and a method `deal_one(self)` that removes one card
  from the end of `self.all_cards` and returns it.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import random

suits = ('Hearts', 'Diamonds', 'Spades', 'Clubs')
ranks = ('Two', 'Three', 'Four', 'Five', 'Six', 'Seven', 'Eight', 'Nine', 'Ten',
         'Jack', 'Queen', 'King', 'Ace')
values = {'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5, 'Six': 6, 'Seven': 7,
          'Eight': 8, 'Nine': 9, 'Ten': 10, 'Jack': 11, 'Queen': 12, 'King': 13,
          'Ace': 14}


class Card:

    def __init__(self, suit, rank):
        self.suit = suit
        self.rank = rank
        self.value = values[rank]

    def __str__(self):
        return self.rank + ' of ' + self.suit


class Deck:

    def __init__(self):
        self.all_cards = []
        for suit in suits:
            for rank in ranks:
                self.all_cards.append(Card(suit, rank))

    def shuffle(self):
        random.shuffle(self.all_cards)

    def deal_one(self):
        return self.all_cards.pop()
```

**Why:** The deck is built by looping over the suits and the ranks and creating one Card for every combination, exactly like the warm-up project. `deal_one` removes the card from the end of the list with `pop()`.

</details>

In [ ]:
import random

deck = Deck()
assert len(deck.all_cards) == 52
assert all(isinstance(card, Card) for card in deck.all_cards)
assert len(set((card.suit, card.rank) for card in deck.all_cards)) == 52

assert str(Card('Hearts', 'Two')) == 'Two of Hearts'
assert Card('Clubs', 'Ace').value == 14
assert Card('Spades', 'Jack').value == 11
assert Card('Diamonds', 'King').value == 13

original = list(deck.all_cards)
random.seed(0)
deck.shuffle()
assert len(deck.all_cards) == 52
assert len(set((card.suit, card.rank) for card in deck.all_cards)) == 52
assert deck.all_cards != original

dealt = deck.deal_one()
assert isinstance(dealt, Card)
assert len(deck.all_cards) == 51

## Q10. Coding exercise (2 points)

Create the timer helpers for the game:

* Import `time` at the top of the cell.
* Create a function **`format_time(seconds)`** that returns the number of seconds as
  a string of the form `MM:SS`, with the minutes and the seconds padded to two
  digits, so `format_time(185)` returns `'03:05'` and `format_time(65)` returns
  `'01:05'`.
* Create a function **`elapsed_seconds(start, end=None)`** that returns the number of
  seconds between `start` and `end`. When `end` is `None` the function uses
  `time.time()` as the end value.
* Create a function **`countdown(seconds, sleep_func=time.sleep)`** that counts down
  from `seconds` to 1. Before every step it calls `sleep_func(1)`, and it returns the
  list of the values it counted down through, so `countdown(3)` returns `[3, 2, 1]`.
  The `sleep_func` argument exists so that the countdown can be tested without
  waiting.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import time


def format_time(seconds):
    minutes, remainder = divmod(int(seconds), 60)
    return '%02d:%02d' % (minutes, remainder)


def elapsed_seconds(start, end=None):
    if end is None:
        end = time.time()
    return end - start


def countdown(seconds, sleep_func=time.sleep):
    values = []
    remaining = int(seconds)
    while remaining > 0:
        values.append(remaining)
        sleep_func(1)
        remaining -= 1
    return values
```

**Why:** The sleep function and the end time are injected so the timer can be unit tested deterministically, and `divmod` gives the minutes and the remaining seconds of the formatted clock.

</details>

In [ ]:
import time

assert format_time(0) == '00:00'
assert format_time(59) == '00:59'
assert format_time(65) == '01:05'
assert format_time(185) == '03:05'

assert elapsed_seconds(100.0, 130.5) == 30.5
assert elapsed_seconds(1000.0) > 0

calls = []
assert countdown(3, sleep_func=calls.append) == [3, 2, 1]
assert calls == [1, 1, 1]
assert countdown(0, sleep_func=calls.append) == []

## Topics covered in this test

* The rules of the War game from the warm-up project
* Modelling `Card`, `Deck` and `Player` classes with OOP
* The blackjack steps of the walkthrough workbook: `Card`, `Deck`, `Hand` and `Chips`
* Timer design: function or class, loop or thread, and how to make it testable
* Unit testing with `unittest` from the errors and exception handling section
* Output formats and error handling with `try` / `except` / `else` / `finally`
* Extension ideas for the milestone project